In [6]:
%run "./data_quality_notebook"

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 18, Finished, Available, Finished, True)

📊 SILVER_SALES VERİ KALİTE RAPORU

📌 Toplam kayıt sayısı: 2823
📌 Kolon sayısı: 25
📌 Kolonlar: ['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

🔍 NULL DEĞER KONTROLÜ:
+-----------+---------------+---------+---------------+-----+---------+------+------+--------+-------+-----------+----+-----------+------------+-----+------------+------------+----+-----+----------+-------+---------+---------------+----------------+--------+
|ORDERNUMBER|QUANTITYORDERED|PRICEEACH|ORDERLINENUMBER|SALES|ORDERDATE|STATUS|QTR_ID|MONTH_ID|YEAR_ID|PRODUCTLINE|MSRP|PRODUCTCODE|CUSTOMERNAME|PHONE|ADDRESSLINE1|ADDRESSLINE2|CITY|STATE|POSTALCODE|COUNTRY|TERRITORY|CONTACTLASTNAME|CONTACTFIRSTNAME|DEALSIZE|
+-----------+---------------+--

SynapseWidget(Synapse.DataFrame, 6f2a6253-f382-4cae-b93b-1ca3c2fd370f)

🔧 OTOMATİK DÜZELTME & AKSİYON KATMANI

✅ DÜZELTMELER:
   ADDRESSLINE2: 0 kayıt → 'N/A' ile dolduruldu
   STATE: 0 kayıt → 'N/A' ile dolduruldu
   POSTALCODE: 0 kayıt → '00000' ile dolduruldu

✅ Düzeltilmiş veri silver_sales'e kaydedildi!

🚨 KRİTİK HATA KONTROLÜ
✅ Tüm kritik kontroller geçti!
✅ Pipeline devam edebilir! 🚀
🧪 UNIT TEST SUITE — RetailMart360

📌 SILVER_SALES TESTLERİ:
✅ PASSED: silver_sales boş değil
✅ PASSED: Kolon sayısı doğru (25 kolon)
✅ PASSED: Kritik kolonlar mevcut
✅ PASSED: Tüm SALES değerleri pozitif
✅ PASSED: ADDRESSLINE2 NULL kalmadı
✅ PASSED: STATE NULL kalmadı
✅ PASSED: POSTALCODE NULL kalmadı

📌 DIM_CUSTOMER_SCD2 TESTLERİ:
✅ PASSED: dim_customer_scd2 boş değil
✅ PASSED: is_current kolonu mevcut
✅ PASSED: Her müşteri için tek aktif SCD2 kaydı var

📌 DIM_GEOGRAPHY_COUNTRY TESTLERİ:
✅ PASSED: dim_geography_country 19 ülke içeriyor
✅ PASSED: Singapore APAC territory'sinde

📊 TEST ÖZETI:
✅ Passed: 12/12
❌ Failed: 0/12

🎉 TÜM TESTLER BAŞARILI — Pipeline devam edebili

In [7]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
#reading the silver table
df = spark.read.table("silver_sales") 
print(f"Satır: {df.count()}, Sütun: {len(df.columns)}")
df.printSchema()

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 19, Finished, Available, Finished, False)

Satır: 2823, Sütun: 25
root
 |-- ORDERNUMBER: integer (nullable = true)
 |-- QUANTITYORDERED: integer (nullable = true)
 |-- PRICEEACH: double (nullable = true)
 |-- ORDERLINENUMBER: integer (nullable = true)
 |-- SALES: double (nullable = true)
 |-- ORDERDATE: date (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- QTR_ID: integer (nullable = true)
 |-- MONTH_ID: integer (nullable = true)
 |-- YEAR_ID: integer (nullable = true)
 |-- PRODUCTLINE: string (nullable = true)
 |-- MSRP: integer (nullable = true)
 |-- PRODUCTCODE: string (nullable = true)
 |-- CUSTOMERNAME: string (nullable = true)
 |-- PHONE: string (nullable = true)
 |-- ADDRESSLINE1: string (nullable = true)
 |-- ADDRESSLINE2: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- POSTALCODE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- TERRITORY: string (nullable = true)
 |-- CONTACTLASTNAME: string (nullable = true)
 |-- CONTACTFIRSTNAME: 

In [8]:
display(df) #this is our silver_sales table

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9e397920-c6aa-48b2-bac5-0fb056cf08a1)

In [9]:
#creating our dimension tables
from pyspark.sql.functions import monotonically_increasing_id, first, when, col, lit

# 1. dim_customer
dim_customer = df.select(
    "CUSTOMERNAME", "PHONE", 
    "CONTACTFIRSTNAME", "CONTACTLASTNAME"
).dropDuplicates()\
 .withColumn("customer_key", monotonically_increasing_id())

# 2. dim_product
dim_product = df.select(
    "PRODUCTCODE", "PRODUCTLINE", "MSRP"
).dropDuplicates()\
 .withColumn("product_key", monotonically_increasing_id())

# 3. dim_geography — temiz, unique, APAC düzeltmeli
dim_geography = spark.read.table("silver_sales")\
    .select("COUNTRY", "TERRITORY")\
    .groupBy("COUNTRY")\
    .agg(first("TERRITORY").alias("TERRITORY"))\
    .withColumn("TERRITORY",
        when(col("COUNTRY") == "SINGAPORE", "APAC")
        .otherwise(col("TERRITORY")))\
    .withColumn("CITY", lit(None).cast("string"))\
    .withColumn("STATE", lit(None).cast("string"))\
    .withColumn("POSTALCODE", lit(None).cast("string"))\
    .withColumn("geography_key", col("COUNTRY"))

dim_geography.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("dim_geography")

print(f"✅ dim_geography oluşturuldu: {dim_geography.count()} ülke")
display(dim_geography)

# 4. dim_date
dim_date = df.select(
    "ORDERDATE", "YEAR_ID", 
    "MONTH_ID", "QTR_ID"
).dropDuplicates()\
 .withColumn("date_key", monotonically_increasing_id())

print(f"dim_customer: {dim_customer.count()} satır")
print(f"dim_product: {dim_product.count()} satır")
print(f"dim_geography: {dim_geography.count()} satır")
print(f"dim_date: {dim_date.count()} satır")

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 21, Finished, Available, Finished, False)

✅ dim_geography oluşturuldu: 19 ülke


SynapseWidget(Synapse.DataFrame, c85da2eb-8544-48d2-8e10-baa87bfeae4a)

dim_customer: 92 satır
dim_product: 109 satır
dim_geography: 19 satır
dim_date: 252 satır


In [10]:
from pyspark.sql.functions import monotonically_increasing_id

# FACT TABLOSU — Star Schema'nın merkezi
# Bu tablo "ne oldu, ne kadar, kaç tane" sorularını cevaplar
# Dimension tablolarına bağlanacak foreign key'leri içerir

fact_sales = df.select(
    # Sipariş kimlik bilgileri
    "ORDERNUMBER",        # Her siparişin benzersiz ID'si
    "ORDERLINENUMBER",    # Sipariş satır numarası
    
    # Ölçümler (metrics) — bunlar aggregate edilebilir sayılar
    "QUANTITYORDERED",    # Kaç adet sipariş verildi
    "PRICEEACH",          # Birim fiyat
    "SALES",              # Toplam satış tutarı
    "DEALSIZE",           # Anlaşma büyüklüğü (Small/Medium/Large)
    
    # Dimension tablolarına bağlantı için kolonlar
    "CUSTOMERNAME",       # dim_customer'a bağlanacak
    "PRODUCTCODE",        # dim_product'a bağlanacak
    "COUNTRY",            # dim_geography'e bağlanacak
    "ORDERDATE",          # dim_date'e bağlanacak
    "STATUS"              # Sipariş durumu
)

# Fact tablosuna unique ID ekle
fact_sales = fact_sales.withColumn(
    "fact_key", monotonically_increasing_id()
)

print(f"fact_sales: {fact_sales.count()} satır, {len(fact_sales.columns)} sütun")
display(fact_sales.limit(5))

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 22, Finished, Available, Finished, False)

fact_sales: 2823 satır, 12 sütun


SynapseWidget(Synapse.DataFrame, 1ddf72d4-c789-4b14-83d8-895a2a1e3c5d)

In [11]:
# Tüm Star Schema tablolarını Delta formatında kaydet
# Delta formatı: ACID uyumlu, versiyonlanabilir, enterprise standart

# Dimension tabloları — Silver katmanına kaydet
dim_customer.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_customer")
# Açıklama: 92 benzersiz müşteri kaydedildi

dim_product.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_product")
# Açıklama: 109 benzersiz ürün kaydedildi

dim_geography.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_geography")
# Açıklama: 79 benzersiz lokasyon kaydedildi

dim_date.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("dim_date")
# Açıklama: 252 benzersiz tarih kaydedildi

# Fact tablosu — Gold katmanına kaydet
# Çünkü fact tablosu iş analizine hazır, aggregate edilebilir
fact_sales.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fact_sales")
# Açıklama: 2823 satır sipariş verisi kaydedildi

print("✅ Tüm Star Schema tabloları başarıyla kaydedildi!")
print("dim_customer, dim_product, dim_geography, dim_date, fact_sales")

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 23, Finished, Available, Finished, False)

✅ Tüm Star Schema tabloları başarıyla kaydedildi!
dim_customer, dim_product, dim_geography, dim_date, fact_sales


In [12]:
%run "./scd_notebook"

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 36, Finished, Available, Finished, True)

Toplam müşteri sayısı: 92
+--------------------+------------+
|        CUSTOMERNAME|       PHONE|
+--------------------+------------+
|Dragon Souveniers...|+65 221 7555|
+--------------------+------------+

+--------------------+------------+
|        CUSTOMERNAME|       PHONE|
+--------------------+------------+
|Dragon Souveniers...|+65 221 9999|
+--------------------+------------+

✅ SCD Type 1 tamamlandı — eski numara silindi, yeni numara yazıldı!
dim_customer_scd2 oluşturuldu: 92 satır


SynapseWidget(Synapse.DataFrame, 18b2fdcc-adb0-4474-b45b-d3df91bc56cc)

+--------------------+---------------+----------+----------+----------+
|        CUSTOMERNAME|          PHONE|is_current|start_date|  end_date|
+--------------------+---------------+----------+----------+----------+
|Dragon Souveniers...|   +65 221 9999|     false|2026-04-15|2026-04-15|
|Dragon Souveniers...|+60 3 2999 8888|      true|2026-04-15|      NULL|
+--------------------+---------------+----------+----------+----------+

✅ SCD Type 2 tamamlandı — tarihsel kayıt korundu!
Joined tablo: 2823 satır


SynapseWidget(Synapse.DataFrame, 08f94e7c-0819-4744-af81-1d5bf74753ca)

...

Unique ülke sayısı: 19


SynapseWidget(Synapse.DataFrame, c5c101cd-f6f3-426b-983c-a88f3279ce79)

Singapore → APAC olarak düzeltildi ✅


SynapseWidget(Synapse.DataFrame, b3d59fed-55a8-4ae4-9d22-3a991265e9e9)

=== HAFTA 3 — MILESTONE 2 ÖZET ===
dim_customer_scd2: 93 kayıt
gold_country_ranking: 19 kayıt
gold_monthly_trend: 29 kayıt
dim_geography_country: 19 kayıt
=== TÜM TABLOLAR HAZIR ✅ ===
⚡ DELTA LAKE OPTİMİZASYON BAŞLIYOR

📌 ADIM 1: OPTIMIZE
✅ silver_sales optimize edildi
✅ dim_customer_scd2 optimize edildi
✅ gold_country_ranking optimize edildi
✅ gold_monthly_trend optimize edildi
✅ dim_geography_country optimize edildi
✅ gold_data_quality_log optimize edildi

📌 ADIM 2: VACUUM
✅ silver_sales vacuum tamamlandı
✅ dim_customer_scd2 vacuum tamamlandı
✅ gold_country_ranking vacuum tamamlandı
✅ gold_monthly_trend vacuum tamamlandı
✅ dim_geography_country vacuum tamamlandı
✅ gold_data_quality_log vacuum tamamlandı

🎉 OPTİMİZASYON TAMAMLANDI!


In [13]:
%run "./optimization_notebook"

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 37, Finished, Available, Finished, True)

⚡ DELTA LAKE OPTİMİZASYON BAŞLIYOR

📌 ADIM 1: OPTIMIZE
✅ silver_sales optimize edildi
✅ dim_customer_scd2 optimize edildi
✅ gold_country_ranking optimize edildi
✅ gold_monthly_trend optimize edildi
✅ dim_geography_country optimize edildi
✅ gold_data_quality_log optimize edildi

📌 ADIM 2: VACUUM
✅ silver_sales vacuum tamamlandı
✅ dim_customer_scd2 vacuum tamamlandı
✅ gold_country_ranking vacuum tamamlandı
✅ gold_monthly_trend vacuum tamamlandı
✅ dim_geography_country vacuum tamamlandı
✅ gold_data_quality_log vacuum tamamlandı

🎉 OPTİMİZASYON TAMAMLANDI!


In [14]:
%run "./anomaly_notebook"

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 38, Finished, Available, Finished, True)

✅ gold_sales_anomalies güncellendi! Anomali sayısı: 28


In [15]:
#%run "./error_notification_notebook"

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 39, Finished, Available, Finished, True)

🚨 PIPELINE HATA VERDİ!
⏰ Hata zamanı: 2026-04-15 23:42:09
📋 Pipeline: EliteArchitect_Pipeline
🔍 Kontrol et: Monitor → Pipeline runs
✅ Hata logu kaydedildi!


In [16]:
# Hangi tablolar var kontrol et
#spark.sql("SHOW TABLES").show()

StatementMeta(, f16cf558-2f78-4f78-ad59-360d455439b2, 40, Finished, Available, Finished, False)